# Lab: Probability & Uncertainty in AI

## Case Study: AI System for Detecting Forged Ancient Manuscripts

---

### Scenario

You are developing an AI system for a **Digital Archaeology Lab**.

Historians must determine whether ancient manuscripts are **genuine or forged**, but evidence is often incomplete and uncertain.

The AI system analyzes multiple uncertain signals:

- Ink composition
- Writing style consistency
- Material age
- Linguistic authenticity
- Discovery context

Using probability theory, the system estimates the likelihood of forgery.

---

## Random Variables

Each variable represents possible worlds (ω):

- Ink (I) = {natural, synthetic}
- Style (S) = {consistent, inconsistent}
- Age (A) = {ancient, modern}
- Language (L) = {authentic, anomalous}
- Context (C) = {controlled, suspicious}
- Authenticity (T) = {genuine, forged}

---

## Tasks

- Compute marginal probabilities
- Evaluate conditional probabilities
- Apply Bayes’ Rule
- Analyze joint probabilities
- Test independence
- Validate probability distributions

---

## Input Files

ink.txt  
style.txt  
age.txt  
language.txt  
context.txt  
authenticity.txt  
joint.txt  

---

## Learning Objectives

- Represent uncertainty in code
- Work with probability distributions
- Apply Bayes’ Rule in real scenarios
- Perform marginalization and conditioning
- Analyze dependencies between variables

## Step 1: Build the Evidence Model

Create a class to store and manage probability data.

In [1]:
class ManuscriptModel:
    def __init__(self):
        self.variables = {}
        self.joint_data = []

    def create_variable(self, name):
        self.variables[name] = {}

    def add_state(self, var, state, prob):
        if var not in self.variables:
            raise ValueError(f"Variable {var} not found")

        self.variables[var][state] = float(prob)

    def get_prob(self, var, state):
        if var not in self.variables:
            return 0
        return self.variables[var].get(state, 0)

    def get_states(self, var):
        if var not in self.variables:
            return []
        return list(self.variables[var].keys())

    def add_joint_entry(self, ink, style, age, authenticity, prob):
        # State as dictionary
        entry = {
            "ink": ink,
            "style": style,
            "age": age,
            "authenticity": authenticity,
            "prob": float(prob)
        }
        self.joint_data.append(entry)

    def get_joint(self):
        return self.joint_data


    def validate_variable(self, var):
        if var not in self.variables:
            return False

        total = sum(self.variables[var].values())
        return abs(total - 1.0) < 1e-6

    def validate_all(self):
        for var in self.variables:
            if not self.validate_variable(var):
                raise ValueError(f"Invalid distribution for {var}")

        total_joint = sum(entry["prob"] for entry in self.joint_data)
        if abs(total_joint - 1.0) > 1e-6:
            raise ValueError("Joint distribution does not sum to 1")

## Step 2: Input Parsers

In [2]:
def load_distribution(file, var_name, model):
    with open(file, 'r') as f:
        for line in f:
            line = line.strip()

            if not line:
                continue

            state, prob = line.split(",")
            model.add_state(var_name, state.strip(), float(prob))


def load_joint(file, model):
    """
    ink,style,age,authenticity,prob
    """
    with open(file, 'r') as f:
        for line in f:
            line = line.strip()

            if not line:
                continue

            ink, style, age, auth, prob = line.split(",")

            model.add_joint_entry(
                ink.strip(),
                style.strip(),
                age.strip(),
                auth.strip(),
                float(prob)
            )

## Step 3: Probability Reasoning Engine

In [3]:
class InferenceEngine:
    def __init__(self, model):
        self.model = model


    def complement(self, p):
        return 1 - p

    def union(self, p_a, p_b, p_ab):
        return p_a + p_b - p_ab


    def filter_joint(self, conditions):
        """
        Return matching joint rows
        """
        results = []

        for row in self.model.get_joint():
            match = True
            for key, value in conditions.items():
                if row.get(key) != value:
                    match = False
                    break
            if match:
                results.append(row)

        return results  

    def compute_joint_prob(self, conditions):
        rows = self.filter_joint(conditions)
        return sum(row["prob"] for row in rows)

    def marginalize(self, var, state):
        return self.compute_joint_prob({var: state})



    def conditional_prob(self, query, evidence):
        combined = {}
        combined.update(query)
        combined.update(evidence)

        p_q_and_e = self.compute_joint_prob(combined)
        p_e = self.compute_joint_prob(evidence)

        if p_e == 0:
            return 0
        return p_q_and_e / p_e

    def bayes_rule(self, p_a, p_b_given_a, p_b):
        if p_b == 0:
            return 0
        return (p_b_given_a * p_a) / p_b

    def check_independence(self, p_a, p_b, p_ab):
        return abs(p_ab - (p_a * p_b)) < 1e-6

## Step 4: Analysis Module

In [4]:
class Analysis:
    def __init__(self, model):
        self.model = model
        self.engine = InferenceEngine(model)

    def prob_forged(self):
        return self.engine.marginalize("authenticity", "forged")

    def forged_given_synthetic_ink(self):
        return self.engine.conditional_prob(
            {"authenticity": "forged"},
            {"ink": "synthetic"}
        )

    def bayes_modern_given_forged(self):
        p_a = self.engine.marginalize("age", "modern")

        p_t = self.engine.marginalize("authenticity", "forged")

        p_t_given_a = self.engine.conditional_prob(
            {"authenticity": "forged"},
            {"age": "modern"}
        )

        return self.engine.bayes_rule(p_a, p_t_given_a, p_t)

    def check_ink_style_independence(self):
        p_i = self.engine.marginalize("ink", "synthetic")

        p_s = self.engine.marginalize("style", "inconsistent")

        p_is = self.engine.compute_joint_prob({
            "ink": "synthetic",
            "style": "inconsistent"
        })

        return self.engine.check_independence(p_i, p_s, p_is)

    def marginal_age(self):
        return {
            "ancient": self.engine.marginalize("age", "ancient"),
            "modern": self.engine.marginalize("age", "modern")
        }

    def suspicious_union(self):
        p_i = self.engine.marginalize("ink", "synthetic")

        p_s = self.engine.marginalize("style", "inconsistent")

        p_is = self.engine.compute_joint_prob({
            "ink": "synthetic",
            "style": "inconsistent"
        })

        return self.engine.union(p_i, p_s, p_is)

## Step 5: Output Writer (Do NOT modify)

In [5]:
def save_results(results, filename="output.txt"):
    text = "Manuscript Authenticity Analysis\n"
    text += "--------------------------------\n"

    for r in results:
        text += r + "\n"

    print(text)

    with open(filename, "w") as f:
        f.write(text)

## Step 6: Exercises

In [ ]:
# Q1: P(Forged | Style = inconsistent)

def forged_given_style(model):
    engine = InferenceEngine(model)
    return engine.conditional_prob(
        {"authenticity": "forged"},
        {"style": "inconsistent"}
    )


# Q2: Check independence of Age and Ink

def age_ink_independent(model):
    engine = InferenceEngine(model)

    # P(A = modern)
    p_a = engine.marginalize("age", "modern")

    # P(I = synthetic)
    p_i = engine.marginalize("ink", "synthetic")

    # P(A ∩ I)
    p_ai = engine.compute_joint_prob({
        "age": "modern",
        "ink": "synthetic"
    })

    return engine.check_independence(p_a, p_i, p_ai)


# Q3:
# If more manuscripts are forged, what happens to P(genuine)?
# Since P(genuine) + P(forged) = 1,
# if P(forged) increases, P(genuine) decreases.


# Q4: Find MOST suspicious combination

def most_suspicious(model):
    max_prob = -1
    best_case = None

    for row in model.get_joint():
        if row["authenticity"] == "forged":
            if row["prob"] > max_prob:
                max_prob = row["prob"]
                best_case = row

    return best_case


# Q5: Compute P(Forged) via marginalization

def forged_marginal(model):
    engine = InferenceEngine(model)
    return engine.marginalize("authenticity", "forged")


# Q6: Normalize a distribution

def normalize(dist):
    total = sum(dist.values())

    if total == 0:
        return dist

    for key in dist:
        dist[key] /= total

    return dist


# Q7: Detect invalid variables
def invalid_variables(model):
    invalid = []

    for var in model.variables:
        if not model.validate_variable(var):
            invalid.append(var)

    return invalid


# Q8: Verify joint total
def joint_total(model):
    return sum(entry["prob"] for entry in model.get_joint())

## Step 7: Main Execution

In [7]:
def run_analysis(model):
    analysis = Analysis(model)

    results = []

    try:
        results.append("P(Forged): " + str(analysis.prob_forged()))
    except:
        results.append("P(Forged): NOT IMPLEMENTED")

    try:
        results.append("P(Forged | Synthetic Ink): " + str(analysis.forged_given_synthetic_ink()))
    except:
        results.append("Conditional Forgery: NOT IMPLEMENTED")

    try:
        results.append("Bayes Age|Forgery: " + str(analysis.bayes_modern_given_forged()))
    except:
        results.append("Bayes: NOT IMPLEMENTED")

    try:
        results.append("Ink ⟂ Style: " + str(analysis.check_ink_style_independence()))
    except:
        results.append("Independence: NOT IMPLEMENTED")

    return results


def main():
    model = ManuscriptModel()

    # Create variables
    for var in ["ink", "style", "age", "language", "context", "authenticity"]:
        model.create_variable(var)

    # Load data
    load_distribution("ink.txt", "ink", model)
    load_distribution("style.txt", "style", model)
    load_distribution("age.txt", "age", model)
    load_distribution("language.txt", "language", model)
    load_distribution("context.txt", "context", model)
    load_distribution("authenticity.txt", "authenticity", model)

    load_joint("joint.txt", model)

    model.validate_all()

    results = run_analysis(model)

    # Exercises
    try:
        results.append("Forged | Style: " + str(forged_given_style(model)))
    except:
        results.append("Q1 NOT DONE")

    try:
        results.append("Age ⟂ Ink: " + str(age_ink_independent(model)))
    except:
        results.append("Q2 NOT DONE")

    try:
        results.append("Most Suspicious: " + str(most_suspicious(model)))
    except:
        results.append("Q4 NOT DONE")

    try:
        results.append("Forged Marginal: " + str(forged_marginal(model)))
    except:
        results.append("Q5 NOT DONE")

    try:
        results.append("Invalid Vars: " + str(invalid_variables(model)))
    except:
        results.append("Q7 NOT DONE")

    try:
        results.append("Joint Sum: " + str(joint_total(model)))
    except:
        results.append("Q8 NOT DONE")

    save_results(results)


if __name__ == "__main__":
    main()

Manuscript Authenticity Analysis
--------------------------------
P(Forged): 0.65
P(Forged | Synthetic Ink): 0.8076923076923077
Bayes Age|Forgery: 0.6153846153846154
Ink ⟂ Style: False
Forged | Style: 0.7692307692307693
Age ⟂ Ink: False
Most Suspicious: {'ink': 'synthetic', 'style': 'inconsistent', 'age': 'modern', 'authenticity': 'forged', 'prob': 0.16}
Forged Marginal: 0.65
Invalid Vars: []
Joint Sum: 1.0

